# SME Credit Scoring — модельный отчёт

**Модели:** Logistic Regression · CatBoost · LightGBM  
**Источник:** результаты `ml/scoring/train_scoring.py` — одна тестовая выборка для всех моделей.

## 1. Путь к эксперименту и зависимости

Нужны `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `ipykernel`. CatBoost/LightGBM для анализа сохранённых предсказаний **не нужны**.

По умолчанию ожидается `ml/scoring/data/compare_42`. Для другого пути задайте переменную окружения `SCORING_RUN_DIR` или измените значение `RUN_DIR` в ячейке. Настройка `SAVE_FIGURES=True` сохраняет PNG рядом с результатами эксперимента — они должны оставаться за пределами Git.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, brier_score_loss,
)

MODEL_NAMES = {"logistic": "Logistic Regression", "catboost": "CatBoost", "lightgbm": "LightGBM"}
SCORE_COL = "predicted_risk_score_demo"
LABEL_COL = "default_12m"
SAVE_FIGURES = True

# Find repository root when launched from repo root or nested project directories.
candidates = [Path.cwd(), *Path.cwd().parents]
repo = next((p for p in candidates if (p / "ml" / "scoring").is_dir()), None)
if repo is None:
    raise RuntimeError("Не найден корень репозитория с ml/scoring/. Откройте Jupyter из корня проекта.")

RUN_DIR = Path(os.environ.get("SCORING_RUN_DIR", str(repo / "ml" / "scoring" / "data" / "compare_42"))).expanduser().resolve()
if not (RUN_DIR / "comparison.csv").is_file():
    raise FileNotFoundError(
        f"Нет {RUN_DIR / 'comparison.csv'}. Сначала запустите train_scoring.py --model all "
        "или измените RUN_DIR в этой ячейке."
    )
FIG_DIR = RUN_DIR / "analysis_figures"
if SAVE_FIGURES:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

def finish_fig(filename: str):
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(FIG_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

print("Результаты:", RUN_DIR)
print("PNG:", FIG_DIR if SAVE_FIGURES else "не сохраняются")

## 2. Загружаем метрики и проверяем сравнимость выборок

Читаем `comparison.csv`, `comparison.json`, `split_manifest.csv` и индивидуальные `test_predictions.csv`. Проверяем совпадение `application_id`, истинных меток и групп заёмщиков. Нельзя сравнивать модели по разным тестовым объектам.

In [ ]:
comparison = pd.read_csv(RUN_DIR / "comparison.csv")
metadata = json.loads((RUN_DIR / "comparison.json").read_text(encoding="utf-8"))
manifest = pd.read_csv(RUN_DIR / "split_manifest.csv", dtype={"application_id": str, "borrower_id": str})
assert {"train", "val", "test"}.issubset(set(manifest["split"]))
assert not manifest["application_id"].duplicated().any(), "Повторяющиеся заявки в split manifest"
groups = {s: set(g["borrower_id"]) for s, g in manifest.groupby("split")}
assert all(groups[a].isdisjoint(groups[b]) for a in groups for b in groups if a != b), "Утечка borrower_id между сплитами"

preds = {}
for model in MODEL_NAMES:
    filename = RUN_DIR / "models" / model / "test_predictions.csv"
    if not filename.is_file():
        raise FileNotFoundError(f"Не найдены предсказания: {filename}. Запустите --model all.")
    df = pd.read_csv(filename, dtype={"application_id": str, "borrower_id": str})
    required = {"application_id", "borrower_id", LABEL_COL, SCORE_COL}
    assert required.issubset(df.columns), f"Отсутствуют столбцы в {filename}"
    assert not df["application_id"].duplicated().any(), f"Дубли в {model}"
    assert df[SCORE_COL].between(0, 1).all(), f"Оценки риска вне [0,1] у {model}"
    assert df[LABEL_COL].isin([0, 1]).all(), f"Некорректные labels у {model}"
    preds[model] = df.set_index("application_id").sort_index()

reference = preds["logistic"]
test_manifest = manifest.loc[manifest["split"] == "test"].set_index("application_id").sort_index()
assert set(reference.index) == set(test_manifest.index), "Предсказания не соответствуют test из manifest"
assert reference["borrower_id"].equals(test_manifest.loc[reference.index, "borrower_id"])
assert np.array_equal(reference[LABEL_COL].to_numpy(), test_manifest.loc[reference.index, LABEL_COL].to_numpy())
for model, df in preds.items():
    assert df.index.equals(reference.index), f"Разные заявки: {model}"
    assert df["borrower_id"].equals(reference["borrower_id"]), f"Разные заёмщики: {model}"
    assert df[LABEL_COL].equals(reference[LABEL_COL]), f"Разные истинные labels: {model}"

y = reference[LABEL_COL].to_numpy(dtype=int)
assert len(np.unique(y)) == 2, "ROC-AUC требует оба класса"
print(f"Test: {len(y)} заявок; simulated defaults: {y.sum()} ({100*y.mean():.1f}%)")
print("Источники меток:", metadata.get("label_source", "unknown"))
print("Один test и отсутствие пересечения borrower_id: OK")

## 3. Метрики моделей



In [ ]:
test = comparison.loc[comparison["split"] == "test"].copy()
test["model_name"] = test["model"].map(MODEL_NAMES)
cols = ["model_name", "n", "events", "event_rate", "roc_auc", "gini", "pr_auc_average_precision", "brier", "log_loss"]
display(test[cols].sort_values("roc_auc", ascending=False).reset_index(drop=True).round(4))

# Check test metrics match saved predictions (to rounding precision).
for model, df in preds.items():
    row = test.set_index("model").loc[model]
    assert abs(roc_auc_score(y, df[SCORE_COL]) - row["roc_auc"]) < 2e-4
    assert abs(average_precision_score(y, df[SCORE_COL]) - row["pr_auc_average_precision"]) < 2e-4
    assert abs(brier_score_loss(y, df[SCORE_COL]) - row["brier"]) < 2e-4
print("Метрики воспроизводятся по test_predictions.csv: OK")

## 4. ROC-кривые на одном test


In [ ]:
plt.figure(figsize=(8, 5))
for model, df in preds.items():
    fpr, tpr, thresholds = roc_curve(y, df[SCORE_COL])
    auc = roc_auc_score(y, df[SCORE_COL])
    plt.plot(fpr, tpr, label=f"{MODEL_NAMES[model]} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Случайный уровень")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC на общей тестовой выборке")
plt.legend()
plt.grid(alpha=.2)
finish_fig("roc_test.png")

## 5. Precision–Recall кривые


In [ ]:
plt.figure(figsize=(8, 5))
for model, df in preds.items():
    precision, recall, thresholds = precision_recall_curve(y, df[SCORE_COL])
    ap = average_precision_score(y, df[SCORE_COL])
    plt.plot(recall, precision, label=f"{MODEL_NAMES[model]} (AP={ap:.3f})")
plt.axhline(y.mean(), color="gray", linestyle="--", label=f"Доля дефолтов ({y.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall на общей тестовой выборке")
plt.legend()
plt.grid(alpha=.2)
finish_fig("pr_test.png")

## 6. Вероятностная калибровка и Brier score


In [ ]:
def calibration_bins(target: np.ndarray, scores: np.ndarray, n_bins: int = 5) -> pd.DataFrame:
    d = pd.DataFrame({"y": target, "p": scores})
    d["bin"] = pd.qcut(d["p"], q=n_bins, duplicates="drop")
    return d.groupby("bin", observed=True).agg(
        predicted=("p", "mean"), observed=("y", "mean"), n=("y", "size")
    ).reset_index(drop=True)

plt.figure(figsize=(8, 5))
for model, df in preds.items():
    curve = calibration_bins(y, df[SCORE_COL].to_numpy())
    plt.plot(curve["predicted"], curve["observed"], marker="o", label=MODEL_NAMES[model])
plt.plot([0, 1], [0, 1], "--", color="gray", label="Идеальная калибровка")
plt.xlabel("Средний прогноз в корзине")
plt.ylabel("Реальная доля simulated defaults в корзине")
plt.title("Калибровка (5 квантильных корзин, DEMO)")
plt.legend()
plt.grid(alpha=.2)
finish_fig("calibration_test.png")

print("Размеры корзин и частота simulated defaults для Logistic Regression:")
display(calibration_bins(y, preds["logistic"][SCORE_COL].to_numpy()).round(4))

## 7. Диагностика неопределённости сравнения


In [ ]:
rows=[]
for model, r in metadata.get("paired_test_auc_vs_logistic", {}).items():
    ci=r.get("paired_cluster_bootstrap_delta_auc_95ci")
    rows.append({"model": MODEL_NAMES.get(model, model), "AUC difference vs logistic":r.get("auc_delta_vs_logistic"),
                 "CI lower": ci[0] if ci else np.nan, "CI upper":ci[1] if ci else np.nan,
                 "95% CI includes zero": bool(ci and ci[0]<=0<=ci[1])})
if rows:
    display(pd.DataFrame(rows).round(4))
else:
    print("В comparison.json нет парных bootstrap-интервалов. Интерпретация различий ограничена.")

## 8. Важность признаков (только внутри одной модели)


In [ ]:
fi_path = RUN_DIR / "models" / "catboost" / "feature_importance.csv"
if fi_path.is_file():
    importance = pd.read_csv(fi_path)
    assert {"feature", "importance"}.issubset(importance.columns)
    top = importance.nlargest(12, "importance").iloc[::-1]
    plt.figure(figsize=(9, 5))
    plt.barh(top["feature"], top["importance"])
    plt.xlabel("CatBoost feature importance (условная шкала)")
    plt.title("Топ-12 признаков CatBoost")
    finish_fig("catboost_feature_importance.png")
else:
    print("Файл feature_importance.csv не найден — график пропущен.")